In [ ]:
#(Updated: handle <bos>/<eos>/<pad>)
# هاد الي مش سلف اوير توكنيزيشن مش يلي بالداتا بيبربر
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

smiles_path = '/content/drive/MyDrive/Diss/smiles.csv'
vocab_out_path = '/content/drive/MyDrive/Diss/vocab.csv'

df = pd.read_csv(smiles_path)
smiles_list = df['smiles'].astype(str).fillna("").str.strip().tolist()
smiles_list = [s.replace(" ", "") for s in smiles_list if s != ""]

# Special tokens
PAD = "<pad>"
BOS = "<bos>"
EOS = "<eos>"

# Char vocab from data
chars = sorted(set("".join(smiles_list)))

# Build indices: 0..2 reserved
token2idx = {PAD:0, BOS:1, EOS:2}
idx2token = {0:PAD, 1:BOS, 2:EOS}

start = 3
for i, c in enumerate(chars):
    token2idx[c] = start + i
    idx2token[start + i] = c

vocab_df = pd.DataFrame({"index": list(idx2token.keys()), "token": list(idx2token.values())})
vocab_df = vocab_df.sort_values("index").reset_index(drop=True)
vocab_df.to_csv(vocab_out_path, index=False)

print("✅ vocab.csv (with <pad>/<bos>/<eos>) saved to:", vocab_out_path)
print("Vocab size:", len(vocab_df))
print(vocab_df.head(15))


In [ ]:
import pandas as pd
import numpy as np

smiles_path = '/content/drive/MyDrive/Diss/smiles.csv'
vocab_path  = '/content/drive/MyDrive/Diss/vocab.csv'
out_path    = '/content/drive/MyDrive/Diss/tokenized_smiles.csv'

vocab_df = pd.read_csv(vocab_path).sort_values("index").reset_index(drop=True)
token2idx = dict(zip(vocab_df["token"].astype(str), vocab_df["index"].astype(int)))

PAD, BOS, EOS = "<pad>", "<bos>", "<eos>"
assert PAD in token2idx and BOS in token2idx and EOS in token2idx

df = pd.read_csv(smiles_path)
smiles_list = df['smiles'].astype(str).fillna("").str.strip().tolist()
smiles_list = [s.replace(" ", "") for s in smiles_list if s != ""]

lengths = np.array([len(s) for s in smiles_list])
max_len = int(np.percentile(lengths, 99))  # طول يغطي 99%
max_len = max(max_len, 10)

# +2 عشان BOS/EOS
seq_len = max_len + 2

tokenized = np.zeros((len(smiles_list), seq_len), dtype=np.int32)  # PAD=0
unk_chars = set()

for i, s in enumerate(smiles_list):
    tokenized[i, 0] = token2idx[BOS]

    s_cut = s[:max_len]
    for j, ch in enumerate(s_cut, start=1):
        if ch in token2idx:
            tokenized[i, j] = token2idx[ch]
        else:
            unk_chars.add(ch)
            tokenized[i, j] = token2idx[PAD]

    end_pos = min(len(s_cut) + 1, seq_len - 1)
    tokenized[i, end_pos] = token2idx[EOS]

if unk_chars:
    print("⚠️ Unknown chars:", sorted(list(unk_chars))[:50])

pd.DataFrame(tokenized).to_csv(out_path, index=False)
print("✅ tokenized_smiles.csv saved to:", out_path)
print("Shape:", tokenized.shape, "Max token:", tokenized.max(), "Vocab size:", len(vocab_df))
